In [37]:
import re
from pathlib import Path
from typing import List, Dict, Optional

def read_markdown(file_path: str) -> str:
    """
    Read a Markdown file and return its full text.
    """
    return Path(file_path).read_text(encoding="utf-8")


def parse_source(md_text: str) -> Optional[str]:
    """
    Grab the value after a top-level '# Source: ...' line, or None if absent.
    """
    match = re.search(r'^\s*#\s*Source:\s*(.+)$', md_text, flags=re.MULTILINE | re.IGNORECASE)
    return match.group(1).strip() if match else None


def extract_sections(file_path: str) -> Dict[str, object]:
    """
    Extracts:
      - 'single' (#) headings (excluding '# Source') with their combined text
      - 'double' (##) headings with their text only up to the next ## or higher (#)
      - 'triple' (###) headings with their text only up to the next ### or higher (## or #)
    Returns a dict with keys:
      - 'source' : optional source URL/string
      - 'single' : list of {'headings': [...], 'text': '...'}
      - 'double' : list of {'heading': '...', 'text': '...'}
      - 'triple' : list of {'heading': '...', 'text': '...'}
    """
    text = read_markdown(file_path)

    # 1️⃣ Source
    source = parse_source(text)

    # 2️⃣ Single (#) headings
    single_matches = list(re.finditer(r'^(?<!#)# (?!Source:)(.+)$', text, re.MULTILINE))
    single = []
    if single_matches:
        headings, collected_texts = [], []
        for i, m in enumerate(single_matches):
            headings.append(m.group(1).strip())
            start, end = m.end(), single_matches[i + 1].start() if i + 1 < len(single_matches) else len(text)
            # Stop at next heading of ANY level
            block = re.split(r'\n#+ ', text[start:end].strip(), 1)[0].strip()
            collected_texts.append(block)
        single.append({"headings": headings, "text": "\n\n".join(collected_texts)})

    # 3️⃣ Double (##) headings (but skip ### or deeper)
    double_matches = list(re.finditer(r'(?m)^## (?!#)(.+)$', text))
    double = []
    for i, m in enumerate(double_matches):
        heading = m.group(1).strip()
        start, end = m.end(), double_matches[i + 1].start() if i + 1 < len(double_matches) else len(text)
        # Stop at next ### heading or higher (#)
        block = re.split(r'(?m)^### |^(?<!#)# ', text[start:end].strip(), 1)[0].strip()
        double.append({"heading": heading, "text": block})

    # 4️⃣ Triple (###) headings
    triple_matches = list(re.finditer(r'(?m)^### (?!#)(.+)$', text))
    triple = []
    for i, m in enumerate(triple_matches):
        heading = m.group(1).strip()
        start, end = m.end(), triple_matches[i + 1].start() if i + 1 < len(triple_matches) else len(text)
        # Stop at next heading of *same or higher* level (## or #)
        block = re.split(r'(?m)^## |^(?<!#)# ', text[start:end].strip(), 1)[0].strip()
        triple.append({"heading": heading, "text": block})

    return {"source": source, "single": single, "double": double, "triple": triple}


# -------- Example Usage (replace with your file path) --------
if __name__ == "__main__":
    data = extract_sections("/home/ali/AI/1.OdooGenie/data-articles/5-ways-odoo-inventory-automate-supply-chain-management.md")
    print("Source:", data["source"])

    print("\n# Single Hash Sections:")
    for s in data["single"]:
        print("-", s["headings"])
        print("[Text]:", s["text"][:100], "...\n")

    print("## Double Hash Sections:")
    for d in data["double"]:
        print("-", d["heading"])
        print("[Text]:", d["text"][:100], "...\n")

    print("### Triple Hash Sections:")
    for t in data["triple"]:
        print("-", t["heading"])
        print("[Text]:", t["text"][:100], "...\n")


Source: https://hsxtech.net/5-ways-odoo-inventory-automate-supply-chain-management/

# Single Hash Sections:
- ['5 Ways Odoo Inventory Automates Supply Chain Management', '5 Odoo Inventory Strategies That Automate Supply Chain Management']
[Text]: 

Effective supply chain management is required for organizations that want smooth operation, reduce ...

## Double Hash Sections:
- 1. Real-Time Stock Visibility for Smarter Decision Making
[Text]: The most significant supply chain issue is having no idea what is currently in stock. Odoo puts the  ...

- 2. Seamless Integration with Other Odoo Modules
[Text]: Another advantage of Odoo Inventory is how it integrates with the other modules of Odoo, such as Sal ...

- 3. Advanced Forecasting and Demand Planning
[Text]: Demand planning is the most crucial activity of supply chain management. Overstocking implies idle r ...

- 4. Enhanced Efficiency with Barcode Scanning
[Text]:  ...

- 5. Multi-Warehouse Management and Global Supply Chain Coordi

In [38]:
import re
from pathlib import Path
from typing import List, Dict, Optional


def read_markdown(file_path: str) -> str:
    return Path(file_path).read_text(encoding="utf-8")


def parse_source(md_text: str) -> Optional[str]:
    """
    Grab the value after a top-level '# Source: ...' line, or None if absent.
    """
    match = re.search(r'^\s*#\s*Source:\s*(.+)$', md_text, flags=re.MULTILINE | re.IGNORECASE)
    return match.group(1).strip() if match else None


def extract_nested_sections(file_path: str) -> Dict[str, object]:
    """
    Extract:
      - Top-level '# Source:' value (if present)
      - All '##' double-hash headings and their text
      - All '###' triple-hash headings as *children* of the correct '##'

    Output structure:
    {
      "source": str | None,
      "double": [
         {
            "heading": str,
            "text": str,       # text under this ## (excluding ### children)
            "children": [
                {"heading": str, "text": str},
                ...
            ]
         },
         ...
      ]
    }
    """
    text = read_markdown(file_path)
    source = parse_source(text)

    # Regex to find every ## or ### heading
    # Capture level ('##' or '###') and heading text
    pattern = re.compile(r'(?m)^(#{2,3}) (?!#)(.+)$')
    matches = list(pattern.finditer(text))

    sections: List[Dict[str, object]] = []

    for i, m in enumerate(matches):
        level = len(m.group(1))
        heading = m.group(2).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        block = text[start:end].strip()

        if level == 2:
            # Split away any ### children from this block
            split = re.split(r'(?m)^### (?!#)', block, 1)
            clean_text = split[0].strip()
            sections.append({"heading": heading, "text": clean_text, "children": []})
        else:  # level == 3
            # Attach to most recent ## parent
            if not sections:
                # If a triple heading appears before any double, treat it as orphan
                sections.append({"heading": "<no parent>", "text": "", "children": []})
            # Remove higher-level headings from block (none deeper than ### needed)
            clean_block = re.split(r'(?m)^## (?!#)', block, 1)[0].strip()
            sections[-1]["children"].append({"heading": heading, "text": clean_block})

    return {"source": source, "double": sections}


# -------- Example Usage --------
if __name__ == "__main__":
    data = extract_nested_sections("/home/ali/AI/1.OdooGenie/data-articles/5-ways-odoo-inventory-automate-supply-chain-management.md")
    print("Source:", data["source"])

    for d in data["double"]:
        print(f"\n## {d['heading']}")
        if d["text"]:
            print("  [Text]:", d["text"][:100], "...")
        for c in d["children"]:
            print(f"   ### {c['heading']}")
            if c["text"]:
                print("      [Text]:", c["text"][:80], "...")


Source: https://hsxtech.net/5-ways-odoo-inventory-automate-supply-chain-management/

## 1. Real-Time Stock Visibility for Smarter Decision Making
  [Text]: The most significant supply chain issue is having no idea what is currently in stock. Odoo puts the  ...
   ### Data-Driven Decisions
      [Text]: With such current data, you can make business decisions with full knowledge of w ...
   ### Key Benefit:
      [Text]: Eliminates stockout and overstock: Accurate and real-time data guarantees best-i ...

## 2. Seamless Integration with Other Odoo Modules
  [Text]: Another advantage of Odoo Inventory is how it integrates with the other modules of Odoo, such as Sal ...
   ### Complete Visibility and Control
      [Text]: Having all the modules in one platform gives you complete visibility into each s ...
   ### Key Benefit:
      [Text]: Eliminates repetitive work: Odoo modules combined save time and minimize room fo ...

## 3. Advanced Forecasting and Demand Planning
  [Text]: Demand pla